# IMC Prosperity 4: Reproducible Replay Figures

This notebook reads exported research data stored in `../data`. It does not enter results by hand. The figures are generated from the same CSV outputs used during the Round 3 and Round 4 research.

Round 5 source and raw backtest files are not in the recovered archive, so this notebook does not create a Round 5 chart.

In [1]:
import os
from pathlib import Path

notebook_dir = Path.cwd().resolve()
root = notebook_dir.parent if notebook_dir.name == 'notebooks' else notebook_dir
mpl_cache = Path(os.environ.get('MPLCONFIGDIR', '/tmp')) / 'prosperity4-matplotlib'
mpl_cache.mkdir(parents=True, exist_ok=True)
os.environ['MPLCONFIGDIR'] = str(mpl_cache)

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import pandas as pd

data_dir = root / 'data'
figure_dir = root / 'assets' / 'research'
figure_dir.mkdir(parents=True, exist_ok=True)
plt.style.use('seaborn-v0_8-whitegrid')

## Input checks

The four files below are unchanged copies of saved research exports.

In [2]:
inputs = {
    'Round 3 tick timeline': data_dir / 'r3_hydrogel_live_test_timeline.csv',
    'Round 4 decision acceptance': data_dir / 'r4_balanced_vs_maxraw_acceptance.csv',
    'Round 4 decision daily metrics': data_dir / 'r4_balanced_vs_maxraw_metrics.csv',
    'Round 4 cover-rule sweep': data_dir / 'r4_voucher_closer_delay_sweep.csv',
}
for label, path in inputs.items():
    assert path.exists(), f'Missing input: {path}'
    print(f'{label}: {path.name} ({path.stat().st_size:,} bytes)')

Round 3 tick timeline: r3_hydrogel_live_test_timeline.csv (162,987 bytes)
Round 4 decision acceptance: r4_balanced_vs_maxraw_acceptance.csv (3,566 bytes)
Round 4 decision daily metrics: r4_balanced_vs_maxraw_metrics.csv (6,559 bytes)
Round 4 cover-rule sweep: r4_voucher_closer_delay_sweep.csv (3,826 bytes)


## Round 3: HYDROGEL diagnostic replay

This was a diagnostic live-test export. It is not the final combined Round 3 portfolio. It shows why a reasonable fill process was still not enough to justify aggressive HYDROGEL inventory scaling.

In [3]:
r3 = pd.read_csv(inputs['Round 3 tick timeline'])
r3['running_peak'] = r3['total_pnl'].cummax()
r3['drawdown'] = r3['total_pnl'] - r3['running_peak']

fig, axes = plt.subplots(2, 1, figsize=(11, 7), sharex=True, gridspec_kw={'height_ratios': [2, 1]})
axes[0].plot(r3['timestamp'] / 1000, r3['total_pnl'], color='#1f4e79', linewidth=1.8, label='Total PnL')
axes[0].axhline(0, color='#6b7280', linewidth=0.8)
axes[0].set_ylabel('PnL')
axes[0].set_title('Round 3 HYDROGEL diagnostic: PnL path from exported tick data')
axes[0].legend(frameon=False, loc='upper left')

axes[1].step(r3['timestamp'] / 1000, r3['HYDROGEL_PACK_pos'], where='post', color='#b45309', linewidth=1.5)
axes[1].axhline(0, color='#6b7280', linewidth=0.8)
axes[1].set_xlabel('Timestamp (thousands of ticks)')
axes[1].set_ylabel('HYDROGEL position')

fig.tight_layout()
fig.savefig(figure_dir / 'r3_hydrogel_diagnostic.png', dpi=180, bbox_inches='tight')
plt.show()

print({
    'final_pnl': float(r3['total_pnl'].iloc[-1]),
    'maximum_drawdown': float(r3['drawdown'].min()),
    'maximum_abs_position': int(r3['HYDROGEL_PACK_pos'].abs().max()),
    'ticks': int(len(r3)),
})

{'final_pnl': 1310.0, 'maximum_drawdown': -926.4921875, 'maximum_abs_position': 24, 'ticks': 1000}


/var/folders/ln/jcfj_dfd3wsfk53_r56tcqn40000gn/T/ipykernel_63377/2669499460.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Round 4: balanced decision versus maximum-return candidate

This uses the exported decision audit for the final balanced version and the maximum-return reference. Both variants share the first two days. The final-day difference is where the balanced version gives up 2,520 of settled PnL to improve the +200 stress result by 9,480.

In [4]:
decision = pd.read_csv(inputs['Round 4 decision acceptance'])
daily = pd.read_csv(inputs['Round 4 decision daily metrics'])
sources = ['balanced_ship', 'maxraw_ref']
labels = {
    'balanced_ship': 'Balanced final version',
    'maxraw_ref': 'Maximum-return reference',
}
colors = {'balanced_ship': '#1f4e79', 'maxraw_ref': '#b45309'}

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), gridspec_kw={'width_ratios': [1.45, 1]})
for source in sources:
    sample = daily.loc[daily['variant'] == source].sort_values('day')
    cumulative = sample['settle'].cumsum()
    axes[0].plot(sample['day'], cumulative / 1000, marker='o', linewidth=2, color=colors[source], label=labels[source])

axes[0].set_xticks([1, 2, 3])
axes[0].set_xlabel('Replay day')
axes[0].set_ylabel('Cumulative settled PnL (thousands)')
axes[0].set_title('Full-day replay')
axes[0].legend(frameon=False, fontsize=8)

summary = decision.set_index('variant').loc[sources]
x = range(len(sources))
width = 0.36
axes[1].bar([i - width / 2 for i in x], summary['settle_3d'] / 1000, width, color=[colors[s] for s in sources], label='Settled PnL')
axes[1].bar([i + width / 2 for i in x], summary['stress200_3d'] / 1000, width, color='#5f7f6e', label='Stress +200 PnL')
axes[1].set_xticks(list(x), ['Balanced', 'Maximum return'])
axes[1].set_ylabel('Three-day PnL (thousands)')
axes[1].set_title('Return versus stress result')
axes[1].legend(frameon=False, fontsize=8)

fig.suptitle('Round 4: choosing a balanced version over the highest-PnL version', y=1.02)
fig.tight_layout()
fig.savefig(figure_dir / 'r4_balanced_vs_maxraw.png', dpi=180, bbox_inches='tight')
plt.show()

summary[['settle_3d', 'stress200_3d', 'max_mdd_abs']]

/var/folders/ln/jcfj_dfd3wsfk53_r56tcqn40000gn/T/ipykernel_63377/1365344673.py:35: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,settle_3d,stress200_3d,max_mdd_abs
variant,,,
balanced_ship,198915.5,43715.5,16402.0
maxraw_ref,201435.5,34235.5,16402.0


## Round 4: voucher cover-rule sweep

The strategy was tested across a grid of cover delays and cancellation thresholds. The selected `delay70_cancel6` rule was not chosen because it had the highest PnL. It was selected because it passed the full set of daily, drawdown, and stress checks.

In [5]:
sweep = pd.read_csv(inputs['Round 4 cover-rule sweep'])
parameters = sweep['variant'].str.extract(r'delay(?P<delay>\d+)_cancel(?P<cancel>\d+)')
sweep = pd.concat([sweep, parameters], axis=1).dropna(subset=['delay', 'cancel']).copy()
sweep[['delay', 'cancel']] = sweep[['delay', 'cancel']].astype(int)
sweep = sweep.sort_values(['cancel', 'delay'])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.8), sharex=True)
for cancel, subset in sweep.groupby('cancel'):
    axes[0].plot(subset['delay'], subset['settle_sum'] / 1000, marker='o', linewidth=1.6, label=f'cancel {cancel}')
    axes[1].plot(subset['delay'], subset['delta_settle'] / 1000, marker='o', linewidth=1.6, label=f'cancel {cancel}')

selected = sweep.loc[sweep['variant'] == 'delay70_cancel6'].iloc[0]
for axis, value in zip(axes, [selected['settle_sum'] / 1000, selected['delta_settle'] / 1000]):
    axis.scatter([selected['delay']], [value], s=70, color='black', zorder=5, label='selected rule')
    axis.set_xlabel('Cover delay (ticks)')
    axis.legend(frameon=False, fontsize=8)

axes[0].set_ylabel('Three-day settled PnL (thousands)')
axes[0].set_title('Absolute replay PnL')
axes[1].set_ylabel('PnL improvement vs. baseline (thousands)')
axes[1].set_title('Improvement vs. baseline')
fig.suptitle('Round 4 voucher closer: 33 recorded parameter variants', y=1.02)
fig.tight_layout()
fig.savefig(figure_dir / 'r4_voucher_cover_sweep.png', dpi=180, bbox_inches='tight')
plt.show()

sweep.loc[sweep['variant'] == 'delay70_cancel6'].T

/var/folders/ln/jcfj_dfd3wsfk53_r56tcqn40000gn/T/ipykernel_63377/784987382.py:25: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


,4
variant,delay70_cancel6
settle_sum,122769.0
delta_settle,18131.0
delta_post100,17243.0
days_improved,3
worst_day_delta,2081.0
slice_sum,25964.5
slice_ratio,1.035412
mdd_worst,-15772.0
mdd_ratio,1.0


## Round 5 source status

The archive contains Round 5 logs and documented results, but the final source code was lost and the raw event-backtest exports are unavailable. I have not generated a Round 5 chart from manually transcribed totals. If the raw files are found later, this notebook can be extended with the same process.